### Install and import libraries :

In [1]:
%pip install google-cloud-bigquery pandas db-dtypes

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import pandas as pd
import numpy as np

from google.cloud import bigquery

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)

### Connect to BigQuery :

In [2]:
PROJECT_ID = "ecommerce-sales-nabin-2026"

client = bigquery.Client(project=PROJECT_ID)

print("BigQuery connection successful!")
print("Project ID:", client.project)

BigQuery connection successful!
Project ID: ecommerce-sales-nabin-2026


### Check the BigQuery connection :

In [3]:
query = """
SELECT
    CURRENT_DATE() AS current_date,
    CURRENT_TIMESTAMP() AS current_timestamp
"""

df_connection = client.query(query).to_dataframe()

display(df_connection)

,current_date,current_timestamp
0,2026-11-19,2026-11-19 11:23:38.717194+00:00


### List all available datasets :

In [5]:
datasets = list(client.list_datasets())

dataset_records = []

for dataset in datasets:
    dataset_records.append({
        "dataset_id": dataset.dataset_id,
        "project_id": dataset.project
    })

df_datasets = pd.DataFrame(dataset_records)

display(df_datasets)

,dataset_id,project_id
0,devflow_analytics,ecommerce-sales-nabin-2026
1,devflow_raw,ecommerce-sales-nabin-2026
2,devflow_staging,ecommerce-sales-nabin-2026
3,devflow_staging_devflow_analytics,ecommerce-sales-nabin-2026
4,github_activity_analytics,ecommerce-sales-nabin-2026
5,olist_analytics,ecommerce-sales-nabin-2026
6,olist_exports,ecommerce-sales-nabin-2026
7,olist_raw,ecommerce-sales-nabin-2026
8,olist_staging,ecommerce-sales-nabin-2026
9,olist_staging_analytics,ecommerce-sales-nabin-2026


### List all tables :

In [6]:
table_records = []

for dataset in datasets:

    tables = list(client.list_tables(dataset.reference))

    for table_item in tables:

        table = client.get_table(table_item.reference)

        table_records.append({
            "dataset_id": dataset.dataset_id,
            "table_name": table.table_id,
            "table_type": table.table_type,
            "num_rows": table.num_rows,
            "num_columns": len(table.schema)
        })

df_tables = pd.DataFrame(table_records)

display(
    df_tables.sort_values(
        ["dataset_id", "table_name"]
    ).reset_index(drop=True)
)

,dataset_id,table_name,table_type,num_rows,num_columns
0,devflow_analytics,mart_api_performance,TABLE,6,13
1,devflow_analytics,mart_churn_analysis,TABLE,3,12
2,devflow_analytics,mart_experiment_analysis,TABLE,2,11
3,devflow_analytics,mart_feature_adoption,TABLE,7,10
4,devflow_analytics,mart_user_engagement,TABLE,9999,12
5,devflow_raw,api_usage,TABLE,50000,8
6,devflow_raw,experiment_assignments,TABLE,6000,5
7,devflow_raw,experiment_outcomes,TABLE,6000,8
8,devflow_raw,feature_usage,TABLE,20041,6
9,devflow_raw,organizations,TABLE,1500,6


### Inspect all table columns :

In [7]:
column_records = []

for dataset in datasets:

    tables = list(client.list_tables(dataset.reference))

    for table_item in tables:

        table = client.get_table(table_item.reference)

        for field in table.schema:

            column_records.append({
                "dataset_id": dataset.dataset_id,
                "table_name": table.table_id,
                "column_name": field.name,
                "data_type": field.field_type,
                "mode": field.mode
            })

df_columns = pd.DataFrame(column_records)

display(df_columns)

,dataset_id,table_name,column_name,data_type,mode
0,devflow_analytics,mart_api_performance,endpoint,STRING,NULLABLE
1,devflow_analytics,mart_api_performance,total_api_records,INTEGER,NULLABLE
2,devflow_analytics,mart_api_performance,total_users,INTEGER,NULLABLE
3,devflow_analytics,mart_api_performance,total_requests,INTEGER,NULLABLE
4,devflow_analytics,mart_api_performance,successful_requests,INTEGER,NULLABLE
...,...,...,...,...,...
577,olist_staging_staging,stg_products,product_width_cm,INTEGER,NULLABLE
578,olist_staging_staging,stg_sellers,seller_id,STRING,NULLABLE
579,olist_staging_staging,stg_sellers,seller_zip_code_prefix,INTEGER,NULLABLE
580,olist_staging_staging,stg_sellers,seller_city,STRING,NULLABLE


### Identify relevant source columns :

In [8]:
keywords = [
    "user",
    "organization",
    "event",
    "feature",
    "adoption",
    "api",
    "request",
    "sdk",
    "subscription",
    "start_date",
    "end_date",
    "created_at",
    "timestamp",
    "activity"
]

relevant_columns = df_columns[
    df_columns["column_name"].str.lower().apply(
        lambda col: any(
            keyword in col for keyword in keywords
        )
    )
].copy()

display(
    relevant_columns.sort_values(
        ["dataset_id", "table_name", "column_name"]
    ).reset_index(drop=True)
)

,dataset_id,table_name,column_name,data_type,mode
0,devflow_analytics,mart_api_performance,api_health_status,STRING,NULLABLE
1,devflow_analytics,mart_api_performance,failed_requests,INTEGER,NULLABLE
2,devflow_analytics,mart_api_performance,successful_requests,INTEGER,NULLABLE
3,devflow_analytics,mart_api_performance,total_api_records,INTEGER,NULLABLE
4,devflow_analytics,mart_api_performance,total_requests,INTEGER,NULLABLE
...,...,...,...,...,...
191,olist_staging_intermediate,int_order_reviews,order_purchase_timestamp,TIMESTAMP,NULLABLE
192,olist_staging_intermediate,int_order_reviews,review_answer_timestamp,TIMESTAMP,NULLABLE
193,olist_staging_intermediate,int_orders_enriched,order_purchase_timestamp,TIMESTAMP,NULLABLE
194,olist_staging_staging,stg_order_reviews,review_answer_timestamp,TIMESTAMP,NULLABLE


### Save the inventory reports :

In [9]:
output_dir = "../data"

os.makedirs(output_dir, exist_ok=True)

df_datasets.to_csv(
    os.path.join(output_dir, "bigquery_dataset_inventory.csv"),
    index=False
)

df_tables.to_csv(
    os.path.join(output_dir, "bigquery_table_inventory.csv"),
    index=False
)

df_columns.to_csv(
    os.path.join(output_dir, "bigquery_column_inventory.csv"),
    index=False
)

relevant_columns.to_csv(
    os.path.join(output_dir, "relevant_source_columns.csv"),
    index=False
)

print("BigQuery inventory reports saved successfully!")

BigQuery inventory reports saved successfully!


### Final verification :

In [10]:
print("DATASETS")
display(df_datasets)

print("TABLES")
display(df_tables)

print("RELEVANT SOURCE COLUMNS")
display(relevant_columns)

DATASETS


,dataset_id,project_id
0,devflow_analytics,ecommerce-sales-nabin-2026
1,devflow_raw,ecommerce-sales-nabin-2026
2,devflow_staging,ecommerce-sales-nabin-2026
3,devflow_staging_devflow_analytics,ecommerce-sales-nabin-2026
4,github_activity_analytics,ecommerce-sales-nabin-2026
5,olist_analytics,ecommerce-sales-nabin-2026
6,olist_exports,ecommerce-sales-nabin-2026
7,olist_raw,ecommerce-sales-nabin-2026
8,olist_staging,ecommerce-sales-nabin-2026
9,olist_staging_analytics,ecommerce-sales-nabin-2026


TABLES


,dataset_id,table_name,table_type,num_rows,num_columns
0,devflow_analytics,mart_api_performance,TABLE,6,13
1,devflow_analytics,mart_churn_analysis,TABLE,3,12
2,devflow_analytics,mart_experiment_analysis,TABLE,2,11
3,devflow_analytics,mart_feature_adoption,TABLE,7,10
4,devflow_analytics,mart_user_engagement,TABLE,9999,12
5,devflow_raw,api_usage,TABLE,50000,8
6,devflow_raw,experiment_assignments,TABLE,6000,5
7,devflow_raw,experiment_outcomes,TABLE,6000,8
8,devflow_raw,feature_usage,TABLE,20041,6
9,devflow_raw,organizations,TABLE,1500,6


RELEVANT SOURCE COLUMNS


,dataset_id,table_name,column_name,data_type,mode
1,devflow_analytics,mart_api_performance,total_api_records,INTEGER,NULLABLE
2,devflow_analytics,mart_api_performance,total_users,INTEGER,NULLABLE
3,devflow_analytics,mart_api_performance,total_requests,INTEGER,NULLABLE
4,devflow_analytics,mart_api_performance,successful_requests,INTEGER,NULLABLE
5,devflow_analytics,mart_api_performance,failed_requests,INTEGER,NULLABLE
...,...,...,...,...,...
507,olist_staging_intermediate,int_order_reviews,review_answer_timestamp,TIMESTAMP,NULLABLE
508,olist_staging_intermediate,int_order_reviews,order_purchase_timestamp,TIMESTAMP,NULLABLE
520,olist_staging_intermediate,int_orders_enriched,order_purchase_timestamp,TIMESTAMP,NULLABLE
560,olist_staging_staging,stg_order_reviews,review_answer_timestamp,TIMESTAMP,NULLABLE


### Inspect the source table schemas :

#### Define the source dataset

In [11]:
SOURCE_DATASET = "devflow_raw"

SOURCE_TABLES = [
    "users",
    "organizations",
    "product_events",
    "feature_usage",
    "api_usage",
    "subscriptions",
    "experiment_assignments",
    "experiment_outcomes"
]

print("Source dataset:", SOURCE_DATASET)
print("Number of source tables:", len(SOURCE_TABLES))

Source dataset: devflow_raw
Number of source tables: 8


#### Inspect the actual column names and data types

In [12]:
schema_records = []

for table_name in SOURCE_TABLES:

    table_id = f"{PROJECT_ID}.{SOURCE_DATASET}.{table_name}"

    table = client.get_table(table_id)

    for field in table.schema:

        schema_records.append({
            "table_name": table_name,
            "column_name": field.name,
            "data_type": field.field_type,
            "mode": field.mode
        })

df_source_schema = pd.DataFrame(schema_records)

display(
    df_source_schema.sort_values(
        ["table_name", "column_name"]
    ).reset_index(drop=True)
)

,table_name,column_name,data_type,mode
0,api_usage,api_usage_id,INTEGER,NULLABLE
1,api_usage,endpoint,STRING,NULLABLE
2,api_usage,error_count,INTEGER,NULLABLE
3,api_usage,request_count,INTEGER,NULLABLE
4,api_usage,request_timestamp,DATE,NULLABLE
5,api_usage,response_time_ms,FLOAT,NULLABLE
6,api_usage,success_count,INTEGER,NULLABLE
7,api_usage,user_id,INTEGER,NULLABLE
8,experiment_assignments,assigned_at,DATE,NULLABLE
9,experiment_assignments,assignment_id,INTEGER,NULLABLE


#### Display each table's schema separately

In [13]:
for table_name in SOURCE_TABLES:

    print("\n" + "=" * 60)
    print("TABLE:", table_name)
    print("=" * 60)

    display(
        df_source_schema[
            df_source_schema["table_name"] == table_name
        ].reset_index(drop=True)
    )


TABLE: users


,table_name,column_name,data_type,mode
0,users,user_id,INTEGER,NULLABLE
1,users,organization_id,INTEGER,NULLABLE
2,users,signup_date,DATE,NULLABLE
3,users,country,STRING,NULLABLE
4,users,user_role,STRING,NULLABLE
5,users,acquisition_channel,STRING,NULLABLE
6,users,is_verified,INTEGER,NULLABLE



TABLE: organizations


,table_name,column_name,data_type,mode
0,organizations,organization_id,INTEGER,NULLABLE
1,organizations,organization_name,STRING,NULLABLE
2,organizations,industry,STRING,NULLABLE
3,organizations,company_size,STRING,NULLABLE
4,organizations,created_date,DATE,NULLABLE
5,organizations,country,STRING,NULLABLE



TABLE: product_events


,table_name,column_name,data_type,mode
0,product_events,event_id,INTEGER,NULLABLE
1,product_events,user_id,INTEGER,NULLABLE
2,product_events,event_name,STRING,NULLABLE
3,product_events,event_timestamp,DATE,NULLABLE
4,product_events,session_id,STRING,NULLABLE
5,product_events,platform,STRING,NULLABLE
6,product_events,device_type,STRING,NULLABLE
7,product_events,event_properties,STRING,NULLABLE


... (6 outputs left out)


TABLE: experiment_assignments


,table_name,column_name,data_type,mode
0,experiment_assignments,assignment_id,INTEGER,NULLABLE
1,experiment_assignments,experiment_id,STRING,NULLABLE
2,experiment_assignments,user_id,INTEGER,NULLABLE
3,experiment_assignments,variant,STRING,NULLABLE
4,experiment_assignments,assigned_at,DATE,NULLABLE



TABLE: experiment_outcomes


,table_name,column_name,data_type,mode
0,experiment_outcomes,outcome_id,INTEGER,NULLABLE
1,experiment_outcomes,assignment_id,INTEGER,NULLABLE
2,experiment_outcomes,user_id,INTEGER,NULLABLE
3,experiment_outcomes,sdk_integrated,INTEGER,NULLABLE
4,experiment_outcomes,first_api_request,INTEGER,NULLABLE
5,experiment_outcomes,adopted_feature,INTEGER,NULLABLE
6,experiment_outcomes,retained_d7,INTEGER,NULLABLE
7,experiment_outcomes,measured_at,DATE,NULLABLE


#### Check sample records from each source table

In [14]:
sample_data = {}

for table_name in SOURCE_TABLES:

    query = f"""
    SELECT *
    FROM `{PROJECT_ID}.{SOURCE_DATASET}.{table_name}`
    LIMIT 5
    """

    sample_data[table_name] = client.query(query).to_dataframe()

    print("\nTABLE:", table_name)

    display(sample_data[table_name])


TABLE: users


,user_id,organization_id,signup_date,country,user_role,acquisition_channel,is_verified
0,1398,302,2024-01-10,Germany,Admin,Organic,1
1,9381,194,2024-01-18,Singapore,Admin,Paid Ads,1
2,2037,424,2024-01-23,Singapore,Admin,Social Media,0
3,9249,718,2024-01-25,Japan,Admin,Organic,0
4,1565,469,2024-02-02,Japan,Admin,Referral,1



TABLE: organizations


,organization_id,organization_name,industry,company_size,created_date,country
0,23,Organization_23,E-commerce,Medium,2025-07-11,Bangladesh
1,106,Organization_106,E-commerce,Small,2024-01-08,Bangladesh
2,126,Organization_126,E-commerce,Small,2024-06-27,Bangladesh
3,171,Organization_171,E-commerce,Enterprise,2024-12-06,Bangladesh
4,258,Organization_258,E-commerce,Small,2025-02-22,Bangladesh



TABLE: product_events


,event_id,user_id,event_name,event_timestamp,session_id,platform,device_type,event_properties
0,17609,7861,analytics_dashboard_viewed,2024-01-31,SESSION_17609,Web,Desktop,failure
1,49212,2507,analytics_dashboard_viewed,2024-02-20,SESSION_49212,Web,Desktop,feature_interaction
2,19018,8748,analytics_dashboard_viewed,2024-02-21,SESSION_19018,IDE,Desktop,success
3,22238,6657,analytics_dashboard_viewed,2024-02-25,SESSION_22238,Web,Desktop,integration_action
4,4883,1312,analytics_dashboard_viewed,2024-02-27,SESSION_4883,Web,Mobile,page_view


... (6 outputs left out)


TABLE: experiment_assignments


,assignment_id,experiment_id,user_id,variant,assigned_at
0,5440,EXP_FEATURE_ONBOARDING_001,5058,Control,2024-03-31
1,2881,EXP_FEATURE_ONBOARDING_001,1202,Control,2024-04-13
2,4310,EXP_FEATURE_ONBOARDING_001,381,Control,2024-04-15
3,4492,EXP_FEATURE_ONBOARDING_001,3951,Control,2024-04-19
4,4986,EXP_FEATURE_ONBOARDING_001,5776,Control,2024-04-25



TABLE: experiment_outcomes


,outcome_id,assignment_id,user_id,sdk_integrated,first_api_request,adopted_feature,retained_d7,measured_at
0,2881,2881,1202,1,0,1,0,2024-04-20
1,4492,4492,3951,0,1,0,0,2024-04-26
2,1206,1206,7301,0,1,0,0,2024-05-17
3,4465,4465,1398,1,1,1,0,2024-05-18
4,5476,5476,4952,1,1,1,0,2024-05-20


#### Check duplicate user IDs

In [15]:
query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT user_id) AS unique_users,
    COUNTIF(user_id IS NULL) AS null_user_ids
FROM `{PROJECT_ID}.{SOURCE_DATASET}.users`
"""

df_user_check = client.query(query).to_dataframe()

display(df_user_check)

,total_rows,unique_users,null_user_ids
0,10000,10000,0


#### Save the source schema

In [16]:
df_source_schema.to_csv(
    "../data/devflow_source_schema.csv",
    index=False
)

print("Source schema saved successfully!")

Source schema saved successfully!


### Validate primary keys and relationships :

#### Primary-key validation

In [22]:
primary_keys = {
    "users": "user_id",
    "organizations": "organization_id",
    "product_events": "event_id",
    "feature_usage": "usage_id",
    "api_usage": "api_usage_id",
    "subscriptions": "subscription_id",
    "experiment_assignments": "assignment_id",
    "experiment_outcomes": "outcome_id"
}

primary_key_results = []

for table_name, key_column in primary_keys.items():

    query = f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT({key_column}) AS non_null_keys,
        COUNT(DISTINCT {key_column}) AS unique_keys,
        COUNTIF({key_column} IS NULL) AS null_keys
    FROM `{PROJECT_ID}.devflow_raw.{table_name}`
    """

    result = client.query(query).to_dataframe().iloc[0]

    primary_key_results.append({
        "table_name": table_name,
        "primary_key": key_column,
        "total_rows": int(result["total_rows"]),
        "unique_keys": int(result["unique_keys"]),
        "null_keys": int(result["null_keys"]),
        "duplicate_keys": int(
            result["non_null_keys"] - result["unique_keys"]
        )
    })

df_primary_key_check = pd.DataFrame(primary_key_results)

display(df_primary_key_check)

,table_name,primary_key,total_rows,unique_keys,null_keys,duplicate_keys
0,users,user_id,10000,10000,0,0
1,organizations,organization_id,1500,1500,0,0
2,product_events,event_id,80000,80000,0,0
3,feature_usage,usage_id,20041,20041,0,0
4,api_usage,api_usage_id,50000,50000,0,0
5,subscriptions,subscription_id,1500,1500,0,0
6,experiment_assignments,assignment_id,6000,6000,0,0
7,experiment_outcomes,outcome_id,6000,6000,0,0


#### Foreign-key relationship validation

In [23]:
relationship_queries = {
    "users_to_organizations": """
        SELECT COUNT(*) AS orphan_records
        FROM `ecommerce-sales-nabin-2026.devflow_raw.users` u
        LEFT JOIN `ecommerce-sales-nabin-2026.devflow_raw.organizations` o
        ON u.organization_id = o.organization_id
        WHERE u.organization_id IS NOT NULL
        AND o.organization_id IS NULL
    """,

    "events_to_users": """
        SELECT COUNT(*) AS orphan_records
        FROM `ecommerce-sales-nabin-2026.devflow_raw.product_events` e
        LEFT JOIN `ecommerce-sales-nabin-2026.devflow_raw.users` u
        ON e.user_id = u.user_id
        WHERE e.user_id IS NOT NULL
        AND u.user_id IS NULL
    """,

    "feature_usage_to_users": """
        SELECT COUNT(*) AS orphan_records
        FROM `ecommerce-sales-nabin-2026.devflow_raw.feature_usage` f
        LEFT JOIN `ecommerce-sales-nabin-2026.devflow_raw.users` u
        ON f.user_id = u.user_id
        WHERE f.user_id IS NOT NULL
        AND u.user_id IS NULL
    """,

    "api_usage_to_users": """
        SELECT COUNT(*) AS orphan_records
        FROM `ecommerce-sales-nabin-2026.devflow_raw.api_usage` a
        LEFT JOIN `ecommerce-sales-nabin-2026.devflow_raw.users` u
        ON a.user_id = u.user_id
        WHERE a.user_id IS NOT NULL
        AND u.user_id IS NULL
    """,

    "subscriptions_to_organizations": """
        SELECT COUNT(*) AS orphan_records
        FROM `ecommerce-sales-nabin-2026.devflow_raw.subscriptions` s
        LEFT JOIN `ecommerce-sales-nabin-2026.devflow_raw.organizations` o
        ON s.organization_id = o.organization_id
        WHERE s.organization_id IS NOT NULL
        AND o.organization_id IS NULL
    """,

    "assignments_to_users": """
        SELECT COUNT(*) AS orphan_records
        FROM `ecommerce-sales-nabin-2026.devflow_raw.experiment_assignments` a
        LEFT JOIN `ecommerce-sales-nabin-2026.devflow_raw.users` u
        ON a.user_id = u.user_id
        WHERE a.user_id IS NOT NULL
        AND u.user_id IS NULL
    """,

    "outcomes_to_assignments": """
        SELECT COUNT(*) AS orphan_records
        FROM `ecommerce-sales-nabin-2026.devflow_raw.experiment_outcomes` o
        LEFT JOIN `ecommerce-sales-nabin-2026.devflow_raw.experiment_assignments` a
        ON o.assignment_id = a.assignment_id
        WHERE o.assignment_id IS NOT NULL
        AND a.assignment_id IS NULL
    """
}

relationship_results = []

for relationship, query in relationship_queries.items():

    result = client.query(query).to_dataframe()

    relationship_results.append({
        "relationship": relationship,
        "orphan_records": int(result.iloc[0]["orphan_records"])
    })

df_relationship_check = pd.DataFrame(relationship_results)

display(df_relationship_check)

,relationship,orphan_records
0,users_to_organizations,0
1,events_to_users,0
2,feature_usage_to_users,0
3,api_usage_to_users,0
4,subscriptions_to_organizations,0
5,assignments_to_users,0
6,outcomes_to_assignments,0


#### Check multiple subscriptions per organization

In [25]:
query = """
SELECT
    organization_id,
    COUNT(*) AS subscription_count
FROM `ecommerce-sales-nabin-2026.devflow_raw.subscriptions`
GROUP BY organization_id
HAVING COUNT(*) > 1
ORDER BY subscription_count DESC
"""

df_multiple_subscriptions = client.query(query).to_dataframe()

print("Organizations with multiple subscriptions:")
display(df_multiple_subscriptions)

print("Number of organizations with multiple subscriptions:",
      len(df_multiple_subscriptions))

Organizations with multiple subscriptions:


,organization_id,subscription_count


Number of organizations with multiple subscriptions: 0


#### Validate experiment assignments and outcomes

In [26]:
query = """
SELECT
    COUNT(*) AS joined_records,
    COUNTIF(o.assignment_id IS NULL) AS assignments_without_outcomes,
    COUNTIF(
        o.assignment_id IS NOT NULL
        AND a.user_id != o.user_id
    ) AS user_id_mismatches
FROM `ecommerce-sales-nabin-2026.devflow_raw.experiment_assignments` a
LEFT JOIN `ecommerce-sales-nabin-2026.devflow_raw.experiment_outcomes` o
    ON a.assignment_id = o.assignment_id
"""

df_experiment_validation = client.query(query).to_dataframe()

display(df_experiment_validation)

,joined_records,assignments_without_outcomes,user_id_mismatches
0,6000,0,0


In [27]:
query = """
SELECT
    assignment_id,
    COUNT(*) AS outcome_count
FROM `ecommerce-sales-nabin-2026.devflow_raw.experiment_outcomes`
GROUP BY assignment_id
HAVING COUNT(*) > 1
ORDER BY outcome_count DESC
"""

df_duplicate_outcomes = client.query(query).to_dataframe()

display(df_duplicate_outcomes)

,assignment_id,outcome_count


#### Validate date ranges

In [28]:
date_columns = {
    "users": ["signup_date"],
    "organizations": ["created_date"],
    "product_events": ["event_timestamp"],
    "feature_usage": ["first_used_at", "last_used_at"],
    "api_usage": ["request_timestamp"],
    "subscriptions": ["start_date", "end_date"],
    "experiment_assignments": ["assigned_at"],
    "experiment_outcomes": ["measured_at"]
}

date_range_results = []

for table_name, columns in date_columns.items():

    for column in columns:

        query = f"""
        SELECT
            MIN({column}) AS minimum_date,
            MAX({column}) AS maximum_date,
            COUNT(*) AS total_rows,
            COUNTIF({column} IS NULL) AS null_dates
        FROM `{PROJECT_ID}.devflow_raw.{table_name}`
        """

        result = client.query(query).to_dataframe().iloc[0]

        date_range_results.append({
            "table_name": table_name,
            "date_column": column,
            "minimum_date": result["minimum_date"],
            "maximum_date": result["maximum_date"],
            "total_rows": int(result["total_rows"]),
            "null_dates": int(result["null_dates"])
        })

df_date_ranges = pd.DataFrame(date_range_results)

display(df_date_ranges)

,table_name,date_column,minimum_date,maximum_date,total_rows,null_dates
0,users,signup_date,2024-01-08,2026-06-23,10000,0
1,organizations,created_date,2024-01-01,2025-12-31,1500,0
2,product_events,event_timestamp,2024-01-15,2026-11-15,80000,0
3,feature_usage,first_used_at,2024-01-17,2026-11-08,20041,0
4,feature_usage,last_used_at,2024-01-24,2026-11-15,20041,0
5,api_usage,request_timestamp,2024-01-10,2026-11-15,50000,0
6,subscriptions,start_date,2024-01-16,2026-06-02,1500,0
7,subscriptions,end_date,2024-03-03,2026-10-28,1500,1043
8,experiment_assignments,assigned_at,2024-03-05,2026-10-31,6000,0
9,experiment_outcomes,measured_at,2024-03-12,2026-11-07,6000,0


#### Feature adoption analysis

In [33]:
query = """
SELECT
    feature_name,
    COUNT(*) AS usage_records,
    COUNT(DISTINCT user_id) AS unique_users,
    SUM(usage_count) AS total_usage_count,
    MIN(first_used_at) AS first_adoption_date,
    MAX(first_used_at) AS latest_adoption_date
FROM `ecommerce-sales-nabin-2026.devflow_raw.feature_usage`
GROUP BY feature_name
ORDER BY unique_users DESC
"""

df_feature_adoption_summary = client.query(query).to_dataframe()

display(df_feature_adoption_summary)

,feature_name,usage_records,unique_users,total_usage_count,first_adoption_date,latest_adoption_date
0,Analytics Dashboard,2945,2945,737665,2024-01-27,2026-10-17
1,Webhooks,2913,2913,718339,2024-01-23,2026-11-08
2,API Documentation,2873,2873,724121,2024-01-28,2026-10-22
3,Rate Limiting,2869,2869,705089,2024-01-17,2026-10-30
4,SDK Integration,2843,2843,721612,2024-01-24,2026-11-01
5,API Monitoring,2802,2802,700146,2024-01-23,2026-10-31
6,API Authentication,2796,2796,678096,2024-01-22,2026-10-29


### Validate the adoption data :

In [4]:
query = """
WITH user_feature_summary AS (
    SELECT
        user_id,
        COUNT(DISTINCT feature_name) AS adopted_feature_count,
        SUM(usage_count) AS total_usage_count
    FROM `ecommerce-sales-nabin-2026.devflow_raw.feature_usage`
    GROUP BY user_id
)

SELECT
    adopted_feature_count,
    COUNT(*) AS number_of_users,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS user_percentage,
    ROUND(AVG(total_usage_count), 2) AS avg_usage_count
FROM user_feature_summary
GROUP BY adopted_feature_count
ORDER BY adopted_feature_count
"""

df_user_feature_distribution = client.query(query).to_dataframe()

display(df_user_feature_distribution)

,adopted_feature_count,number_of_users,user_percentage,avg_usage_count
0,1,3341,33.41,248.10
1,2,3277,32.77,493.48
2,3,3382,33.82,750.75


In [5]:
query = """
WITH user_feature_summary AS (
    SELECT
        user_id,
        COUNT(DISTINCT feature_name) AS adopted_feature_count,
        SUM(usage_count) AS total_usage_count
    FROM `ecommerce-sales-nabin-2026.devflow_raw.feature_usage`
    GROUP BY user_id
)

SELECT
    CASE
        WHEN ufs.user_id IS NULL THEN 'No Feature Adopted'
        WHEN ufs.adopted_feature_count = 1 THEN '1 Feature'
        WHEN ufs.adopted_feature_count = 2 THEN '2 Features'
        WHEN ufs.adopted_feature_count = 3 THEN '3 Features'
        ELSE 'More than 3 Features'
    END AS adoption_group,

    COUNT(*) AS number_of_users,

    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*)
         FROM `ecommerce-sales-nabin-2026.devflow_raw.users`),
        2
    ) AS user_percentage

FROM `ecommerce-sales-nabin-2026.devflow_raw.users` u
LEFT JOIN user_feature_summary ufs
    ON u.user_id = ufs.user_id

GROUP BY adoption_group
ORDER BY
    CASE adoption_group
        WHEN 'No Feature Adopted' THEN 0
        WHEN '1 Feature' THEN 1
        WHEN '2 Features' THEN 2
        WHEN '3 Features' THEN 3
        ELSE 4
    END
"""

df_all_user_adoption = client.query(query).to_dataframe()

display(df_all_user_adoption)

,adoption_group,number_of_users,user_percentage
0,1 Feature,3341,33.41
1,2 Features,3277,32.77
2,3 Features,3382,33.82


In [6]:
query = """
SELECT
    feature_name,
    FORMAT_DATE('%Y-%m', first_used_at) AS adoption_month,
    COUNT(DISTINCT user_id) AS adopting_users
FROM `ecommerce-sales-nabin-2026.devflow_raw.feature_usage`
GROUP BY
    feature_name,
    adoption_month
ORDER BY
    feature_name,
    adoption_month
"""

df_monthly_feature_adoption = client.query(query).to_dataframe()

display(df_monthly_feature_adoption)

,feature_name,adoption_month,adopting_users
0,API Authentication,2024-01,1
1,API Authentication,2024-02,4
2,API Authentication,2024-03,19
3,API Authentication,2024-04,24
4,API Authentication,2024-05,34
...,...,...,...
235,Webhooks,2026-07,43
236,Webhooks,2026-08,26
237,Webhooks,2026-09,18
238,Webhooks,2026-10,5


In [7]:
query = """
SELECT
    MIN(first_used_at) AS earliest_adoption_date,
    MAX(first_used_at) AS latest_adoption_date,
    COUNT(DISTINCT FORMAT_DATE('%Y-%m', first_used_at)) AS active_months,
    COUNT(DISTINCT feature_name) AS total_features,
    COUNT(*) AS total_feature_usage_records,
    COUNT(DISTINCT user_id) AS unique_users
FROM `ecommerce-sales-nabin-2026.devflow_raw.feature_usage`
"""

df_adoption_coverage = client.query(query).to_dataframe()

display(df_adoption_coverage)

,earliest_adoption_date,latest_adoption_date,active_months,total_features,total_feature_usage_records,unique_users
0,2024-01-17,2026-11-08,35,7,20041,10000


In [8]:
display(
    df_monthly_feature_adoption
    .groupby("feature_name")["adopting_users"]
    .agg(["count", "sum", "mean", "min", "max"])
    .round(2)
    .sort_values("sum", ascending=False)
)

,count,sum,mean,min,max
feature_name,,,,,
Analytics Dashboard,34,2945,86.62,1,143
Webhooks,35,2913,83.23,1,137
API Documentation,34,2873,84.5,1,152
Rate Limiting,34,2869,84.38,2,138
SDK Integration,35,2843,81.23,1,165
API Monitoring,34,2802,82.41,1,142
API Authentication,34,2796,82.24,1,141
